<a href="https://colab.research.google.com/github/Rithvikns/python/blob/main/Framework/Pytorch/project_2/project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.tensorboard import SummaryWriter
import os


In [2]:
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data Augmentation & Normalization
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


Using device: cpu


In [3]:
# Load CIFAR-10 Dataset
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:03<00:00, 48.4MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [4]:
# Define Custom CNN Model with Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super(ResidualBlock, self).__init__()
        stride = 2 if downsample else 1
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2) if downsample else None

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.res1 = ResidualBlock(32, 32)
        self.res2 = ResidualBlock(32, 64, downsample=True)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, 10)
    def forward(self, x):
        x = self.layer1(x)
        x = self.res1(x)
        x = self.res2(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [6]:
# Initialize model, loss, optimizer, and scheduler
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
writer = SummaryWriter("runs/cifar10_experiment")

In [7]:
# Training Loop
def train(epoch):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if i % 100 == 99:  # Log every 100 mini-batches
            print(f'Epoch [{epoch+1}], Step [{i+1}], Loss: {loss.item():.4f}')
            writer.add_scalar('training loss', loss.item(), epoch * len(train_loader) + i)
    scheduler.step()

In [8]:
# Evaluation Loop
def test():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')
    return accuracy

In [9]:
# Checkpoint Saving & Loading
checkpoint_path = "model_checkpoint.pth"
def save_checkpoint():
    torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, checkpoint_path)

def load_checkpoint():
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print("Checkpoint Loaded!")

In [10]:
# Main Execution
load_checkpoint()
num_epochs = 20
for epoch in range(num_epochs):
    train(epoch)
    acc = test()
    writer.add_scalar('accuracy', acc, epoch)
    save_checkpoint()

writer.close()


Epoch [1], Step [100], Loss: 1.7159
Epoch [1], Step [200], Loss: 1.5856
Epoch [1], Step [300], Loss: 1.4735
Epoch [1], Step [400], Loss: 1.5001
Epoch [1], Step [500], Loss: 1.5658
Epoch [1], Step [600], Loss: 1.1946
Epoch [1], Step [700], Loss: 1.2444
Test Accuracy: 47.80%
Epoch [2], Step [100], Loss: 1.1220
Epoch [2], Step [200], Loss: 1.2893
Epoch [2], Step [300], Loss: 1.0502
Epoch [2], Step [400], Loss: 1.2041
Epoch [2], Step [500], Loss: 1.1905
Epoch [2], Step [600], Loss: 1.1229
Epoch [2], Step [700], Loss: 1.1024
Test Accuracy: 56.49%
Epoch [3], Step [100], Loss: 1.0816
Epoch [3], Step [200], Loss: 1.2361
Epoch [3], Step [300], Loss: 0.9395
Epoch [3], Step [400], Loss: 1.0501
Epoch [3], Step [500], Loss: 1.1584
Epoch [3], Step [600], Loss: 1.1359
Epoch [3], Step [700], Loss: 1.0005
Test Accuracy: 57.60%
Epoch [4], Step [100], Loss: 1.2039
Epoch [4], Step [200], Loss: 1.0397
Epoch [4], Step [300], Loss: 1.0432
Epoch [4], Step [400], Loss: 0.9534


KeyboardInterrupt: 